# Actor graph features across agents of one grid (`bus14`)

Chapter 7 measures how the encoder's inputs shift between `bus14` and WCCI.
The same question applies *within* one environment: the GNN encoder and the
action-scoring MLP are **shared across agents**, so every agent's local graph
is drawn from a different region of the grid and feeds the same weights.

Same four metrics as the cross-grid study (`common/feature_stats.compare_samples`),
same do-nothing protocol: 6000 steps, 300 per chronic, corrected construction
with one-hop context on.

In [ ]:
"""Per-agent local-graph feature samples on one grid, do-nothing rollout."""
import json, sys, warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, grid2op
sys.path.insert(0, str(Path("/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task")))
from common.graph import make_grid_graph_builder

TASK = Path("/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task")
OUT  = TASK / "outputs/graph_feature_comparison/per_agent"
STEPS, CAP = 6000, 300

scen = json.loads((TASK / "env/scenario.json").read_text())["environments"]["bus14"]
env = grid2op.make(scen["grid2op_id"])
domains = {f"agent_{i}": s for i, s in enumerate(scen["agent_stations"])}
b = make_grid_graph_builder("bus", env, domains, include_neighbors=True,
                            context_requires_connection=True,
                            structural_relations_controlled_only=True)
node_names, edge_names = b.node_features, b.edge_features
acc = {a: {f"node/{n}": [] for n in node_names} for a in domains}
for a in domains:
    acc[a].update({f"edge/{n}": [] for n in edge_names})

obs, steps, ep, in_ep = env.reset(), 0, 0, 0
while steps < STEPS:
    graphs = b.build(obs)
    for a in domains:
        g = graphs[a]
        nf, ef = np.asarray(g["node_features"]), np.asarray(g["edge_features"])
        nm, em = np.asarray(g["node_mask"]) > 0, np.asarray(g["edge_mask"]) > 0
        for j, n in enumerate(node_names): acc[a][f"node/{n}"].append(nf[nm, j])
        for j, n in enumerate(edge_names): acc[a][f"edge/{n}"].append(ef[em, j])
    obs, _, done, _ = env.step(env.action_space({}))
    steps += 1; in_ep += 1
    if done or in_ep >= CAP:
        obs = env.reset(); ep += 1; in_ep = 0

for a in domains:
    np.savez_compressed(OUT / f"samples_{a}.npz",
                        **{k: np.concatenate(v) for k, v in acc[a].items()})
(OUT / "meta.json").write_text(json.dumps({
    "env_id": "bus14", "steps": steps, "episodes": ep + 1,
    "agents": sorted(domains), "substations": {a: s for a, s in domains.items()},
    "node_feature_names": node_names, "edge_feature_names": edge_names}, indent=2))
print(f"{steps} steps, {ep+1} episodes")
for a in sorted(domains):
    print(f"  {a}: substations {domains[a]}  node rows/step "
          f"{len(np.concatenate(acc[a]['node/gen_p']))/steps:.1f}")


## Loading the samples and computing pairwise shift

Three agents give three pairs. Sparse channels are also reported conditioned
on the nodes that carry the asset, exactly as in the cross-grid notebook.

In [ ]:
import json, sys, itertools
from pathlib import Path
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
TASK = Path("/Users/corentinplumet/Documents/RL_Marl2grid/Topology_Task")
FIG  = Path("/Users/corentinplumet/Documents/RL_Marl2grid/latex/figures")
sys.path.insert(0, str(TASK))
from common.feature_stats import compare_samples

D = TASK / "outputs/graph_feature_comparison/per_agent"
meta = json.loads((D / "meta.json").read_text())
AG = meta["agents"]
S = {a: dict(np.load(D / f"samples_{a}.npz")) for a in AG}
SPARSE = {"gen_p", "gen_theta", "load_p", "load_theta"}
FEATS = ["rho", "load_theta", "gen_p", "load_p", "gen_theta", "domain_mask"]
# cross-grid KS from chapter 7, all sampled entries
CROSS = {"load_theta": 0.331, "gen_theta": 0.115, "rho": 0.141,
         "gen_p": 0.089, "load_p": 0.040}

def nonzero(s):
    return {k: (v[v != 0] if k.split("/")[-1] in SPARSE else v) for k, v in s.items()}

def pairwise(sets):
    out = {}
    for a, b in itertools.combinations(AG, 2):
        for r in compare_samples(sets[a], sets[b], a, b):
            out.setdefault(r["feature"].split("/")[-1], {})[f"{a[-1]}-{b[-1]}"] = r["ks"]
    return out

ks_all, ks_nz = pairwise(S), pairwise({a: nonzero(v) for a, v in S.items()})

### What the pairs show

Every measured channel differs more between two agents of `bus14` than
`bus14` differs from WCCI. Worst pair per feature, against the cross-grid
value from Chapter 7:

| feature | worst inter-agent KS | pair | cross-grid KS | ratio |
|---|---|---|---|---|
| `rho` | **0.393** | 0 vs 2 | 0.141 | **2.8x** |
| `load_theta` | 0.358 | 0 vs 2 | 0.331 | 1.1x |
| `gen_p` | 0.329 | 0 vs 2 | 0.089 | 3.7x |
| `load_p` | 0.240 | 1 vs 2 | 0.040 | **6.0x** |
| `gen_theta` | 0.228 | 0 vs 2 | 0.115 | 2.0x |

Conditioning on the nodes that carry the asset widens it further: `gen_p`
reaches **KS 0.753** between agents 1 and 2.

`line_status` and `timestep_overflow` are constant under a do-nothing policy
and carry no information in either comparison.

## Figure 1 — inter-agent KS against the cross-grid value

The black dashes are the `bus14` vs WCCI KS from Chapter 7.

In [ ]:
# --- figure 1: inter-agent KS against the cross-grid value -------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, (ks, title) in zip(axes, [(ks_all, "all sampled entries"),
                                  (ks_nz, "conditioned on nodes that carry the asset")]):
    pairs = ["0-1", "0-2", "1-2"]
    x = np.arange(len(FEATS)); w = 0.26
    for i, p in enumerate(pairs):
        ax.bar(x + (i - 1) * w, [ks.get(f, {}).get(p, np.nan) for f in FEATS],
               w, label=f"agent {p[0]} vs {p[2]}")
    ax.plot(x, [CROSS.get(f, np.nan) for f in FEATS], "k_", ms=22, mew=2.2,
            label="bus14 vs WCCI")
    ax.set_xticks(x); ax.set_xticklabels(FEATS, rotation=30, ha="right", fontsize=9)
    ax.set_title(title, fontsize=10); ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("KS distance"); axes[0].legend(fontsize=8)
fig.suptitle("Feature shift between agents of one grid, against the cross-grid shift", y=1.0)
fig.tight_layout(); fig.savefig(FIG / "agent_feature_shift_ks.png", dpi=200); plt.close(fig)

## Figure 2 — the distributions themselves

In [ ]:
# --- figure 2: the two channels that matter most -----------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9))
for ax, key, cond in zip(axes, ["edge/rho", "node/gen_p"], [False, True]):
    for a in AG:
        v = S[a][key]
        if cond: v = v[v != 0]
        lo, hi = np.quantile(np.concatenate([S[x][key] for x in AG]), [0.0, 0.995])
        ax.hist(v, bins=60, range=(lo, hi), density=True, histtype="step", lw=1.6,
                label=f"{a} (subs {meta['substations'][a]})")
    ax.set_xlabel(key.split("/")[-1] + (" , nonzero only" if cond else ""))
    ax.set_ylabel("density"); ax.grid(alpha=0.3); ax.legend(fontsize=7)
fig.suptitle("Per-agent input distributions on the same grid", y=1.02)
fig.tight_layout(); fig.savefig(FIG / "agent_feature_shift_dists.png", dpi=200); plt.close(fig)

### `rho` is the one to look at

Chapter 7 singles out `rho` as the channel needing no correction — "already
comparable across grids because it was defined in per-unit terms", KS 0.141.
Between agent 0 and agent 2 **on the same grid** it is 0.393.

Per-unit normalisation makes a quantity comparable across *networks*. It does
nothing about agent 2's six substations sitting in a differently loaded part of
the grid than agent 0's four. The chapter's cleanest example of a well-posed
feature is well-posed only along the axis it was tested on.

## Summary table

In [ ]:
print(f"{meta['steps']} steps, {meta['episodes']} episodes, agents {AG}")
print(f"\n{'feature':<14}{'worst inter-agent KS':>22}{'pair':>7}{'cross-grid KS':>16}{'ratio':>8}")
for f in FEATS:
    if f not in ks_all: continue
    p, k = max(ks_all[f].items(), key=lambda kv: kv[1])
    c = CROSS.get(f)
    print(f"{f:<14}{k:>22.3f}{p:>7}{(f'{c:.3f}' if c else '-'):>16}"
          f"{(f'{k/c:.1f}x' if c else '-'):>8}")

## What this means

The encoder and the action-scoring MLP are **shared across agents**, so this is
the same class of problem as the cross-grid shift, on an axis the thesis has not
measured before — and here it is the larger of the two.

It is also unaddressed by the existing correction. Physical scaling divides by a
per-*environment* constant, so all three agents receive the same divisor and
nothing about the inter-agent gap changes. A per-agent statistic would fix the
scale but would break the weight sharing that makes one encoder serve every
agent, so it is not a free substitution.

**Two caveats.**

1. This is a **do-nothing** rollout. It characterises the input distribution the
   encoder is initialised against, not what a trained policy induces.
2. Part of the gap is structural rather than electrical: agent 2 controls six
   substations against agent 0's four, so the graphs differ in size as well as in
   loading. Splitting the two effects would need agents matched on substation
   count.